Logical planning

Specification and manually derived candidate

I = {At(Robot, A), At(Package, A)}. G = {At(Package, C)}. The connected directed pairs are A-B, B-A, B-C, and C-B.

Move(x,y) requires At(Robot,x), forbids At(Robot,y), removes At(Robot,x), and adds At(Robot,y). PickUp(Package,l) requires the robot and package at l, forbids Holding(Package), removes the package location, and adds Holding(Package). Drop(Package,l) requires the robot at l and Holding(Package), forbids an existing package location at l, removes Holding(Package), and adds At(Package,l). Pickup and drop are generated for all three locations.

Pickup at A is initially applicable because both location facts hold and the holding fact is absent. Drop at C is not applicable because both its holding and robot-location requirements are absent.

| Step | Action | State |
|---|---|---|
| 0 | Initial | At(Robot,A), At(Package,A) |
| 1 | PickUp(Package,A) | At(Robot,A), Holding(Package) |
| 2 | Move(A,B) | At(Robot,B), Holding(Package) |
| 3 | Move(B,C) | At(Robot,C), Holding(Package) |
| 4 | Drop(Package,C) | At(Robot,C), At(Package,C) |

This candidate was derived during this assisted implementation and replayed by the executable transition rules.

Implementation

Operator.available checks positive inclusion and negative disjointness. Operator.execute rejects an inapplicable action, then removes and adds effects. shortest_plan uses BFS over immutable sets, stores predecessor/action links, tests the goal by subset inclusion, and returns None when the reachable frontier is exhausted. An already satisfied goal returns an empty plan. Replay independently checks applicability and the recorded states.

Test results

solvable: found=True; plan=['PickUp(Package, A)', 'Move(A, B)', 'Move(B, C)', 'Drop(Package, C)']; final=['At(Package, C)', 'At(Robot, C)']; verified=True. Initial=['At(Package, A)', 'At(Robot, A)']; goal=['At(Package, C)'].

no_pickup: found=False; plan=None; final=None; verified=None. Initial=['At(Package, A)', 'At(Robot, A)']; goal=['At(Package, C)'].

irrelevant_robot_moves: found=False; plan=None; final=None; verified=None. Initial=['At(Package, A)', 'At(Robot, A)']; goal=['At(Package, C)'].

The irrelevant-actions test offers only robot moves, so the robot reaching C cannot satisfy the package goal. Removing pickup actions makes the delivery impossible. The successful shortest plan has four actions, all replayed. The tests also reject pickup when Holding(Package) is already true, directly testing a negative precondition.

Logic and search

Current state -> check preconditions -> select applicable actions -> generate successor states -> explore alternatives -> check goal.

Logic defines valid transitions. Search decides the order in which sequences of those transitions are explored. Listing an action does not make it executable: its required facts must hold in the current state. Preconditions and effects make the specification explicit before implementing search. Without them, a planner could drop an unheld package or move from a location it never reached. A reasonable-looking plan can therefore fail when actually executed. Codex supplied the code and tests; the trace, minimal length, impossible case, negative preconditions, and package goal were checked through execution. Planning uses the same BFS structure as navigation but searches fact sets rather than grid positions. Explicit state transitions are stronger evidence than a generated prose justification because each transition is checked against executable rules.

Prolog verification

planner.pl contains the directed connectivity facts, movement rules, a complete route rule, and wet_road -> slippery -> reduce_speed. SWI-Prolog executed the conjunction checking can_move(a,b), valid_move(b,c), valid_route([a,b,c]), failure of can_move(a,c), failure of valid_move(a,c), and reduce_speed. It exited successfully. Direct a-c movement fails because no connected(a,c) fact exists; this does not deny the multi-step route through B. A fact is an unconditional clause; a rule derives its head when its body succeeds. A query requests proof using the knowledge base. Prolog provides an independently executed connectivity check, but its movement-only verifier does not verify pickup/drop state semantics. Python replay checks those semantics. Prolog uses unification and backtracking, so its operational behavior is not identical to arbitrary classical logical entailment.

In [1]:
from collections import deque
from dataclasses import dataclass


@dataclass(frozen=True)
class Operator:
    label: str
    required: frozenset
    forbidden: frozenset
    added: frozenset
    removed: frozenset

    def available(self, facts):
        return self.required <= facts and not self.forbidden & facts

    def execute(self, facts):
        if not self.available(facts):
            raise ValueError(f'Preconditions not satisfied: {self.label}')
        return frozenset((facts - self.removed) | self.added)


def shortest_plan(initial, target, operators):
    initial, target = frozenset(initial), frozenset(target)
    pending = deque([initial])
    previous = {initial: None}
    while pending:
        facts = pending.popleft()
        if target <= facts:
            steps = []
            while previous[facts] is not None:
                parent, operator = previous[facts]
                steps.append((operator, facts))
                facts = parent
            return steps[::-1]
        for operator in operators:
            if operator.available(facts):
                successor = operator.execute(facts)
                if successor not in previous:
                    previous[successor] = (facts, operator)
                    pending.append(successor)
    return None


def warehouse_operators():
    operators = []
    for source, destination in [('A', 'B'), ('B', 'A'), ('B', 'C'), ('C', 'B')]:
        before, after = f'At(Robot, {source})', f'At(Robot, {destination})'
        operators.append(Operator(f'Move({source}, {destination})', frozenset([before]),
                                  frozenset([after]), frozenset([after]), frozenset([before])))
    for location in 'ABC':
        robot, package = f'At(Robot, {location})', f'At(Package, {location})'
        holding = 'Holding(Package)'
        operators.append(Operator(f'PickUp(Package, {location})', frozenset([robot, package]),
                                  frozenset([holding]), frozenset([holding]), frozenset([package])))
        operators.append(Operator(f'Drop(Package, {location})', frozenset([robot, holding]),
                                  frozenset([package]), frozenset([package]), frozenset([holding])))
    return operators


def replay(initial, plan):
    facts = frozenset(initial)
    for operator, recorded in plan:
        facts = operator.execute(facts)
        if facts != recorded:
            raise ValueError('Recorded state differs from executed state')
    return facts


def experiment():
    initial = {'At(Robot, A)', 'At(Package, A)'}
    target = {'At(Package, C)'}
    operators = warehouse_operators()
    cases = {'solvable': operators,
             'no_pickup': [op for op in operators if not op.label.startswith('PickUp')],
             'irrelevant_robot_moves': [op for op in operators if op.label.startswith('Move')]}
    results = {}
    for name, choices in cases.items():
        plan = shortest_plan(initial, target, choices)
        final = replay(initial, plan) if plan is not None else None
        results[name] = {'initial': sorted(initial), 'goal': sorted(target),
                         'found': plan is not None,
                         'plan': [op.label for op, _ in plan] if plan is not None else None,
                         'states': [sorted(state) for _, state in plan] if plan is not None else None,
                         'final': sorted(final) if final is not None else None,
                         'valid': target <= final if final is not None else None}
    return results

import json
results = experiment()
print(json.dumps(results, indent=2))


{
  "solvable": {
    "initial": [
      "At(Package, A)",
      "At(Robot, A)"
    ],
    "goal": [
      "At(Package, C)"
    ],
    "found": true,
    "plan": [
      "PickUp(Package, A)",
      "Move(A, B)",
      "Move(B, C)",
      "Drop(Package, C)"
    ],
    "states": [
      [
        "At(Robot, A)",
        "Holding(Package)"
      ],
      [
        "At(Robot, B)",
        "Holding(Package)"
      ],
      [
        "At(Robot, C)",
        "Holding(Package)"
      ],
      [
        "At(Package, C)",
        "At(Robot, C)"
      ]
    ],
    "final": [
      "At(Package, C)",
      "At(Robot, C)"
    ],
    "valid": true
  },
  "no_pickup": {
    "initial": [
      "At(Package, A)",
      "At(Robot, A)"
    ],
    "goal": [
      "At(Package, C)"
    ],
    "found": false,
    "plan": null,
    "states": null,
    "final": null,
    "valid": null
  },
  "irrelevant_robot_moves": {
    "initial": [
      "At(Package, A)",
      "At(Robot, A)"
    ],
    "goal": [
      "At(P

In [2]:
checks = experiment()
assert checks['solvable']['valid']
assert len(checks['solvable']['plan']) == 4
assert not checks['no_pickup']['found']
assert not checks['irrelevant_robot_moves']['found']
initial = {'At(Robot, A)', 'At(Package, A)'}
assert shortest_plan(initial, initial, []) == []
print('Planning checks passed')


Planning checks passed


Prolog knowledge base

```prolog
connected(a, b).
connected(b, a).
connected(b, c).
connected(c, b).
can_move(From, To) :- connected(From, To).
valid_move(From, To) :- can_move(From, To).
valid_route([_]).
valid_route([From, To | Rest]) :- valid_move(From, To), valid_route([To | Rest]).
wet_road.
slippery :- wet_road.
reduce_speed :- slippery.
```